# 데이터 인덱싱 및 선택

[2부](02.00-Introduction-to-NumPy.ipynb)에서는 NumPy 배열의 값에 액세스하고, 설정하고, 수정하는 방법과 도구를 자세히 살펴보았습니다.
여기에는 인덱싱(예: `arr[2, 1]`), 슬라이싱(예: `arr[:, 1:5]`), 마스킹(예: `arr[arr > 0]`), 팬시 인덱싱(예: `arr[0, [1, 5]]`) 및 이들의 조합(예: `arr[:, [1, 5]]`)이 포함됩니다.
여기서는 Pandas 'Series' 및 'DataFrame' 개체의 값에 액세스하고 수정하는 유사한 방법을 살펴보겠습니다.
NumPy 패턴을 사용해 본 적이 있다면 Pandas의 해당 패턴이 매우 친숙하게 느껴질 것입니다. 단, 주의해야 할 몇 가지 단점이 있습니다.

1차원 `Series` 개체의 간단한 사례부터 시작한 다음 더 복잡한 2차원 `DataFrame` 개체로 넘어갑니다.

## 시리즈 데이터 선택

이전 장에서 보았듯이 'Series' 객체는 여러 면에서 1차원 NumPy 배열처럼 작동하고 여러 면에서 표준 파이썬(Python) 사전과 유사하게 작동합니다.
이 두 가지 겹치는 비유를 염두에 두면 이러한 배열의 데이터 인덱싱 및 선택 패턴을 이해하는 데 도움이 됩니다.

### 시리즈를 사전으로 사용

사전과 마찬가지로 'Series' 개체는 키 컬렉션에서 값 컬렉션으로의 매핑을 제공합니다.

In [1]:
import pandas as pd
data = pd.Series([0.25, 0.5, 0.75, 1.0],
                 index=['a', 'b', 'c', 'd'])
data

a    0.25
b    0.50
c    0.75
d    1.00
dtype: float64

In [2]:
data['b']

0.5

또한 사전과 같은 파이썬(Python) 표현식과 메소드를 사용하여 키/인덱스 및 값을 검사할 수 있습니다.

In [3]:
'a' in data

True

In [4]:
data.keys()

Index(['a', 'b', 'c', 'd'], dtype='object')

In [5]:
list(data.items())

[('a', 0.25), ('b', 0.5), ('c', 0.75), ('d', 1.0)]

`Series` 객체는 사전과 같은 구문으로 수정할 수도 있습니다.
새 키에 할당하여 사전을 확장할 수 있는 것처럼 새 인덱스 값에 할당하여 '시리즈'를 확장할 수 있습니다.

In [6]:
data['e'] = 1.25
data

a    0.25
b    0.50
c    0.75
d    1.00
e    1.25
dtype: float64

이러한 객체의 쉬운 변경 가능성은 편리한 기능입니다. 내부적으로 Pandas는 발생해야 할 메모리 레이아웃 및 데이터 복사에 대한 결정을 내리며 사용자는 일반적으로 이러한 문제에 대해 걱정할 필요가 없습니다.

1차원 배열로서의 ### 시리즈

'시리즈'는 사전과 같은 인터페이스를 기반으로 하며 NumPy 배열과 동일한 기본 메커니즘, 즉 슬라이스, 마스킹 및 멋진 인덱싱을 통해 배열 스타일 항목 선택을 제공합니다.
그 예는 다음과 같습니다.

In [7]:
# slicing by explicit index
data['a':'c']

a    0.25
b    0.50
c    0.75
dtype: float64

In [8]:
# slicing by implicit integer index
data[0:2]

a    0.25
b    0.50
dtype: float64

In [9]:
# masking
data[(data > 0.3) & (data < 0.8)]

b    0.50
c    0.75
dtype: float64

In [10]:
# fancy indexing
data[['a', 'e']]

a    0.25
e    1.25
dtype: float64

그중에서도 슬라이싱이 가장 혼란스러운 원인이 될 수 있습니다.
명시적 인덱스(예: `data['a':'c']`)로 슬라이싱할 때 최종 인덱스는 슬라이스에 *포함*되는 반면 암시적 인덱스(예: `data[0:2]`)로 슬라이싱할 때는 최종 인덱스가 슬라이스에서 *제외됩니다*.

### 인덱서: loc 및 iloc

`Series`에 명시적인 정수 인덱스가 있는 경우 `data[1]`과 같은 인덱싱 작업은 명시적인 인덱스를 사용하고 `data[1:3]`과 같은 슬라이싱 작업은 암시적인 파이썬(Python) 스타일 인덱스를 사용합니다.

In [11]:
data = pd.Series(['a', 'b', 'c'], index=[1, 3, 5])
data

1    a
3    b
5    c
dtype: object

In [12]:
# explicit index when indexing
data[1]

'a'

In [13]:
# implicit index when slicing
data[1:3]

3    b
5    c
dtype: object

정수 인덱스의 경우 이러한 잠재적인 혼란 때문에 Pandas는 특정 인덱싱 체계를 명시적으로 노출하는 몇 가지 특별한 *indexer* 속성을 제공합니다.
이는 함수형 메서드가 아니라 '시리즈'의 데이터에 특정 슬라이싱 인터페이스를 노출하는 속성입니다.

첫째, `loc` 속성은 항상 명시적인 인덱스를 참조하는 인덱싱 및 슬라이싱을 허용합니다.

In [14]:
data.loc[1]

'a'

In [15]:
data.loc[1:3]

1    a
3    b
dtype: object

`iloc` 속성은 항상 암시적 파이썬(Python) 스타일 인덱스를 참조하는 인덱싱 및 슬라이싱을 허용합니다.

In [16]:
data.iloc[1]

'b'

In [17]:
data.iloc[1:3]

3    b
5    c
dtype: object

파이썬(Python) 코드의 기본 원칙 중 하나는 "명시적인 것이 암시적인 것보다 낫다"는 것입니다.
`loc`과 `iloc`의 명시적 특성은 깨끗하고 읽기 쉬운 코드를 유지하는 데 도움이 됩니다. 특히 정수 인덱스의 경우 일관되게 사용하면 혼합 인덱스/슬라이싱 규칙으로 인한 미묘한 버그를 방지할 수 있습니다.

## DataFrame에서 데이터 선택

`DataFrame`은 2차원 또는 구조화된 배열과 같은 다양한 방식으로 작동하고 다른 방식으로는 동일한 인덱스를 공유하는 `Series` 구조의 사전처럼 작동한다는 점을 기억하세요.
이러한 비유는 이 구조 내에서 데이터 선택을 탐색할 때 염두에 두는 데 도움이 될 수 있습니다.

### 사전으로서의 DataFrame

우리가 고려할 첫 번째 비유는 관련 `Series` 개체의 사전인 `DataFrame`입니다.
주의 지역과 인구에 대한 예로 돌아가 보겠습니다.

In [18]:
area = pd.Series({'California': 423967, 'Texas': 695662,
                  'Florida': 170312, 'New York': 141297,
                  'Pennsylvania': 119280})
pop = pd.Series({'California': 39538223, 'Texas': 29145505,
                 'Florida': 21538187, 'New York': 20201249,
                 'Pennsylvania': 13002700})
data = pd.DataFrame({'area':area, 'pop':pop})
data

,area,pop
California,423967,39538223
Texas,695662,29145505
Florida,170312,21538187
New York,141297,20201249
Pennsylvania,119280,13002700


'DataFrame'의 열을 구성하는 개별 'Series'는 열 이름의 사전 스타일 인덱싱을 통해 액세스할 수 있습니다.

In [19]:
data['area']

California      423967
Texas           695662
Florida         170312
New York        141297
Pennsylvania    119280
Name: area, dtype: int64

마찬가지로 문자열인 열 이름에 속성 스타일 액세스를 사용할 수 있습니다.

In [20]:
data.area

California      423967
Texas           695662
Florida         170312
New York        141297
Pennsylvania    119280
Name: area, dtype: int64

이는 유용한 약어이지만 모든 경우에 적용되는 것은 아니라는 점을 명심하세요!
예를 들어 열 이름이 문자열이 아니거나 열 이름이 'DataFrame'의 메서드와 충돌하는 경우 이 속성 스타일 액세스는 불가능합니다.
예를 들어 `DataFrame`에는 `pop` 메서드가 있으므로 `data.pop`은 `pop` 열이 아닌 이를 가리킵니다.

In [21]:
data.pop is data["pop"]

False

특히, 속성을 통해 열 할당을 시도하려는 유혹을 피해야 합니다(즉, `data.pop = z` 대신 `data['pop'] = z` 사용).

앞서 설명한 `Series` 개체와 마찬가지로 이 사전 스타일 구문을 사용하여 개체를 수정할 수도 있습니다. 이 경우 새 열을 추가합니다.

In [22]:
data['density'] = data['pop'] / data['area']
data

,area,pop,density
California,423967,39538223,93.257784
Texas,695662,29145505,41.896072
Florida,170312,21538187,126.463121
New York,141297,20201249,142.970120
Pennsylvania,119280,13002700,109.009893


이는 `Series` 객체 간의 요소별 산술의 간단한 구문을 미리 보여줍니다. 이에 대해서는 [Pandas의 데이터 작업](03.03-Operations-in-Pandas.ipynb)에서 자세히 살펴보겠습니다.

### 2차원 배열로서의 DataFrame

이전에 언급했듯이 `DataFrame`을 향상된 2차원 배열로 볼 수도 있습니다.
`values` 속성을 사용하여 원시 기본 데이터 배열을 검사할 수 있습니다.

In [23]:
data.values

array([[4.23967000e+05, 3.95382230e+07, 9.32577842e+01],
       [6.95662000e+05, 2.91455050e+07, 4.18960717e+01],
       [1.70312000e+05, 2.15381870e+07, 1.26463121e+02],
       [1.41297000e+05, 2.02012490e+07, 1.42970120e+02],
       [1.19280000e+05, 1.30027000e+07, 1.09009893e+02]])

이 그림을 염두에 두면 `DataFrame` 자체에서 친숙한 배열과 유사한 많은 작업을 수행할 수 있습니다.
예를 들어 전체 `DataFrame`을 전치하여 행과 열을 바꿀 수 있습니다.

In [24]:
data.T

,California,Texas,Florida,New York,Pennsylvania
area,4.239670e+05,6.956620e+05,1.703120e+05,1.412970e+05,1.192800e+05
pop,3.953822e+07,2.914550e+07,2.153819e+07,2.020125e+07,1.300270e+07
density,9.325778e+01,4.189607e+01,1.264631e+02,1.429701e+02,1.090099e+02


그러나 `DataFrame` 객체의 인덱싱에 관해서는 사전 스타일의 열 인덱싱이 이를 단순히 NumPy 배열로 처리하는 능력을 배제한다는 것이 분명합니다.
특히 단일 인덱스를 배열에 전달하면 행에 액세스됩니다.

In [25]:
data.values[0]

array([4.23967000e+05, 3.95382230e+07, 9.32577842e+01])

단일 "인덱스"를 `DataFrame`에 전달하면 열에 액세스합니다.

In [26]:
data['area']

California      423967
Texas           695662
Florida         170312
New York        141297
Pennsylvania    119280
Name: area, dtype: int64

따라서 배열 스타일 인덱싱에는 또 다른 규칙이 필요합니다.
여기서 Pandas는 앞서 언급한 'loc' 및 'iloc' 인덱서를 다시 사용합니다.
`iloc` 인덱서를 사용하면 기본 배열을 단순한 NumPy 배열인 것처럼 인덱싱할 수 있지만(암시적 파이썬(Python) 스타일 인덱스 사용) `DataFrame` 인덱스와 열 레이블은 결과에 유지됩니다.

In [27]:
data.iloc[:3, :2]

,area,pop
California,423967,39538223
Texas,695662,29145505
Florida,170312,21538187


마찬가지로 `loc` 인덱서를 사용하면 배열과 같은 스타일로 기본 데이터를 인덱싱할 수 있지만 명시적인 인덱스와 열 이름을 사용합니다.

In [28]:
data.loc[:'Florida', :'pop']

,area,pop
California,423967,39538223
Texas,695662,29145505
Florida,170312,21538187


익숙한 NumPy 스타일 데이터 액세스 패턴을 이러한 인덱서 내에서 사용할 수 있습니다.
예를 들어 `loc` 인덱서에서는 다음과 같이 마스킹과 팬시 인덱싱을 결합할 수 있습니다.

In [29]:
data.loc[data.density > 120, ['pop', 'density']]

,pop,density
Florida,21538187,126.463121
New York,20201249,142.970120


이러한 인덱싱 규칙 중 하나를 사용하여 값을 설정하거나 수정할 수도 있습니다. 이는 NumPy 작업에 익숙할 수 있는 표준 방식으로 수행됩니다.

In [30]:
data.iloc[0, 2] = 90
data

,area,pop,density
California,423967,39538223,90.000000
Texas,695662,29145505,41.896072
Florida,170312,21538187,126.463121
New York,141297,20201249,142.970120
Pennsylvania,119280,13002700,109.009893


Pandas 데이터 조작에 대한 능숙도를 높이려면 간단한 `DataFrame`을 사용하여 시간을 보내고 이러한 다양한 인덱싱 접근 방식에서 허용되는 인덱싱, 슬라이싱, 마스킹 및 멋진 인덱싱 유형을 탐색하는 것이 좋습니다.

### 추가 색인 생성 규칙

이전 논의와 상충되는 것처럼 보일 수 있지만 그럼에도 불구하고 실제로 유용할 수 있는 몇 가지 추가 색인 작성 규칙이 있습니다.
첫째, *인덱싱*은 열을 참조하는 반면, *슬라이싱*은 행을 참조합니다.

In [31]:
data['Florida':'New York']

,area,pop,density
Florida,170312,21538187,126.463121
New York,141297,20201249,142.970120


이러한 조각은 인덱스가 아닌 숫자로 행을 참조할 수도 있습니다.

In [32]:
data[1:3]

,area,pop,density
Texas,695662,29145505,41.896072
Florida,170312,21538187,126.463121


마찬가지로 직접 마스킹 작업은 열 단위가 아닌 행 단위로 해석됩니다.

In [33]:
data[data.density > 120]

,area,pop,density
Florida,170312,21538187,126.463121
New York,141297,20201249,142.970120


이 두 가지 규칙은 구문론적으로 NumPy 배열의 규칙과 유사하며 Pandas 규칙의 틀에 정확히 맞지 않을 수 있지만 실용적인 유용성 때문에 포함되었습니다.